In [ ]:
!pip install -q -U gigachat transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.2 MB/s eta 0:00:00


In [ ]:
import json
import re
import time

from google.colab import files, userdata
from gigachat import GigaChat
from transformers import AutoTokenizer

In [ ]:
GIGACHAT_KEY = userdata.get("GIGACHAT_CREDENTIALS")

giga = GigaChat(
    credentials=GIGACHAT_KEY,
    scope="GIGACHAT_API_PERS",
    base_url="https://api.giga.chat/v1",
    model="GigaChat-2-Pro",

    # Для Colab-пилота
    verify_ssl_certs=False,

    # Чтобы длинный запрос не падал по timeout
    timeout=180,
    max_retries=3,
    retry_backoff_factor=1.0,
)

print("GigaChat подключён ✅")

GigaChat подключён ✅


In [ ]:
uploaded = files.upload()

filename = next(iter(uploaded))

with open(filename, "r", encoding="utf-8") as f:
    fight = json.load(f)

print("Fight ID:", fight.get("id"))
print("Раундов:", len(fight["rounds"]))
print("Есть human_votes:", "human_votes" in fight)
print("Есть result:", "result" in fight)

Saving fub_video_transcript.json to fub_video_transcript.json
Fight ID: 1
Раундов: 3
Есть human_votes: True
Есть result: True


In [ ]:
inference_data = {
    "id": fight.get("id"),
    "situation": fight["situation"],
    "rounds": fight["rounds"],
}

ground_truth = {
    "human_votes": fight.get("human_votes"),
    "result": fight.get("result"),
}

print("В inference идут только:")
print(inference_data.keys())

В inference идут только:
dict_keys(['id', 'situation', 'rounds'])


In [ ]:
SYSTEM_PROMPT = """
Сожми один раунд управленческого поединка для последующей
самостоятельной оценки другой моделью.

Ты НЕ являешься судьёй.

Не определяй победителя.
Не оценивай игроков.
Не сравнивай их как сильнее/слабее, лучше/хуже,
эффективнее/неэффективнее, убедительнее/неубедительнее.

Сохраняй только факты переговоров:

- предложения;
- контрпредложения;
- согласия и отказы;
- обязательства;
- условия;
- требования;
- уступки;
- угрозы и альтернативы;
- важные вопросы и ответы;
- существенные изменения предмета переговоров;
- суммы, сроки и проценты.

Не добавляй факты, которых нет в стенограмме.

Не превращай:
- предложение в соглашение;
- условное согласие в окончательное;
- намерение в выполненное действие.

Любое существенное предложение или контрпредложение,
содержащее сумму денег, срок или процент,
обязательно сохрани.

Верни только JSON следующего вида:

{
  "round_id": 1,
  "key_events": [
    "краткое фактическое событие",
    "краткое фактическое событие"
  ],
  "unresolved": [
    "нерешённый вопрос"
  ]
}

Не более 8 key_events.

Каждый key_event — максимум одно короткое предложение.

Не добавляй Markdown и текст вне JSON.
"""
SYSTEM_PROMPT += """

Если в стенограмме есть вопрос и непосредственный ответ на него,
и места недостаточно для обоих, сохраняй итоговый ответ,
решение или обязательство, а не сам вопрос.

Особенно точно сохраняй модальность высказываний.
Фразы вида "если", "допустим", "возможно",
"готов рассмотреть" не являются окончательным согласием.
Не описывай их словами "согласился" или "принял".
"""

In [ ]:
def parse_json_response(text):
    text = text.strip()

    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\s*```$",
        "",
        text
    )

    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1:
        raise ValueError("JSON не найден")

    return json.loads(
        text[start:end + 1]
    )

In [ ]:
def compress_round(round_data):

    round_input = {
        "round_id": round_data["round_id"],
        "roles": round_data["roles"],
        "turns": round_data["turns"],
    }

    user_prompt = f"""
ОБЩАЯ СИТУАЦИЯ:

{fight["situation"]}


РАУНД:

{json.dumps(
    round_input,
    ensure_ascii=False,
    indent=2
)}
"""

    response = giga.chat({
        "model": "GigaChat-2-Pro",

        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],

        "temperature": 0.1,
        "max_tokens": 1200,
    })

    raw_output = response.choices[0].message.content

    result = parse_json_response(raw_output)

    # round_id берём из исходника, а не доверяем LLM
    result["round_id"] = round_data["round_id"]

    return result, response

In [ ]:
compressed_rounds = []
responses = []

for rnd in fight["rounds"]:

    print(
        f"Сжимаем round {rnd['round_id']}..."
    )

    result, response = compress_round(rnd)

    compressed_rounds.append(result)
    responses.append(response)

    print(
        f"✅ round {rnd['round_id']}: "
        f"{len(result.get('key_events', []))} событий"
    )

    time.sleep(1)

print("\nВсе раунды обработаны ✅")

Сжимаем round 1...
✅ round 1: 3 событий
Сжимаем round 2...
✅ round 2: 5 событий
Сжимаем round 3...
✅ round 3: 8 событий

Все раунды обработаны ✅


In [ ]:
print(
    json.dumps(
        compressed_rounds,
        ensure_ascii=False,
        indent=2
    )
)

[
  {
    "round_id": 1,
    "key_events": [
      "Промтех предлагает выплатить Синхрону 50 млн рублей взамен на отказ от суда и претензий.",
      "Синхрон согласен отказаться от претензий по текущему заказу при выплате 50 млн рублей.",
      "Промтех обещает произвести выплату сегодня."
    ],
    "unresolved": []
  },
  {
    "round_id": 2,
    "key_events": [
      "Директору Даталинк необходима выплата 50 млн рублей для предотвращения кассового разрыва.",
      "Директор по безопасности предложил провести анализ текущих обязательств Даталинк перед сотрудниками и контрагентами.",
      "Даталинк согласился подготовить подробный список обязательных платежей на три месяца.",
      "Директор по безопасности пообещал перераспределить средства Промтеха для покрытия срочных обязательств Даталинк.",
      "Обещан ответ по финансированию после предоставления списка обязательств Даталинком."
    ],
    "unresolved": [
      "Нет информации о сроках и суммах возможного финансирования Промте

In [ ]:
for rnd in compressed_rounds:

    assert "round_id" in rnd
    assert "key_events" in rnd
    assert "unresolved" in rnd

    assert len(rnd["key_events"]) <= 8

print("Базовая структура корректна ✅")

Базовая структура корректна ✅


In [ ]:
summary_by_round = {
    rnd["round_id"]: rnd
    for rnd in compressed_rounds
}


decision_state = {
    "situation": fight["situation"],
    "rounds": []
}


for original_round in fight["rounds"]:

    round_id = original_round["round_id"]

    summary = summary_by_round[round_id]

    decision_state["rounds"].append({
        "round_id": round_id,

        "roles": original_round["roles"],

        "key_events":
            summary.get("key_events", []),

        "unresolved":
            summary.get("unresolved", [])
    })


print(
    json.dumps(
        decision_state,
        ensure_ascii=False,
        indent=2
    )
)

{
  "situation": "Ситуация 15. Внутренний приоритет. Предприятие Даталинк занимается разработкой цифровых решений для крупного холдинга Азимут. Одним из ключевых внутренних заказчиков Даталинк является завод Промтех. Обе структуры входят в состав холдинга Азимут и подчиняются единому акционеру. Параллельно с этим Промтех более пяти лет сотрудничает с внешним IT-подрядчиком компанией Синхрон, которая в холдинг не входит. Из-за сложных внутренних регламентов и организационных процедур Промтех на протяжении всех лет задерживала оплату разработки на 3-4 месяца. Подрядчики привыкли к этому графику, однако в текущем году возникла критическая задолженность Промтеха перед обеими IT-компаниями сразу за 9 месяцев. Общая сумма долга составила 100 млн рублей, ровно по 50 млн перед Даталинк и перед компанией Синхрон. У Промтеха появилась финансовая возможность погасить задолженность частично на сумму 50 млн рублей. Руководство Промтеха намерено направить все деньги внешнему подрядчику Синхрон, чтоб

In [ ]:
decision_text_check = json.dumps(
    decision_state,
    ensure_ascii=False
)

assert "human_votes" not in decision_text_check
assert "result" not in decision_text_check
assert "judge_id" not in decision_text_check

print("Ground truth в Decision Model не попадает ✅")

Ground truth в Decision Model не попадает ✅


In [ ]:
decision_tokenizer = AutoTokenizer.from_pretrained(
    "StrandsAgents/strands-decider-2B-hobson-v21"
)

print("Tokenizer Strands загружен ✅")

Tokenizer Strands загружен ✅


In [ ]:
decision_text = json.dumps(
    decision_state,
    ensure_ascii=False,
    separators=(",", ":")
)

token_ids = decision_tokenizer.encode(
    decision_text,
    add_special_tokens=False
)

state_tokens = len(token_ids)

print(
    "Размер decision_state:",
    state_tokens,
    "токенов"
)

if state_tokens <= 3000:
    print("✅ В предварительный бюджет помещается")
else:
    print("❌ Нужно сильнее сжимать")

Размер decision_state: 935 токенов
✅ В предварительный бюджет помещается


In [ ]:
for rnd, response in zip(
    fight["rounds"],
    responses
):

    print(
        f"Round {rnd['round_id']}:",
        response.usage
    )

Round 1: prompt_tokens=1087 completion_tokens=79 total_tokens=1166 precached_prompt_tokens=0
Round 2: prompt_tokens=1941 completion_tokens=160 total_tokens=2101 precached_prompt_tokens=621
Round 3: prompt_tokens=1412 completion_tokens=220 total_tokens=1632 precached_prompt_tokens=621


In [ ]:
with open(
    "decision_state.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        decision_state,
        f,
        ensure_ascii=False,
        indent=2
    )

print("decision_state.json сохранён ✅")

decision_state.json сохранён ✅


In [ ]:
!pip install -q -U strands-decider
!pip install -q -U "torchao>=0.16.0"

In [ ]:
import subprocess
import time
import requests
import os

MODEL_ID = "StrandsAgents/strands-decider-2B-hobson-v21"

os.system("pkill -f 'strands-decider serve'")
time.sleep(2)

log_file = open("strands_server.log", "w")

server = subprocess.Popen(
    [
        "strands-decider",
        "serve",
        MODEL_ID,
        "--device", "cuda",
        "--port", "8000",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print("Процесс запущен, PID:", server.pid)

Процесс запущен, PID: 4437


In [ ]:
ready = False

for i in range(90):

    if server.poll() is not None:
        print(
            "❌ Strands завершился с кодом:",
            server.returncode
        )
        break

    try:
        r = requests.get(
            "http://127.0.0.1:8000/health",
            timeout=2
        )

        if r.ok:
            print("✅ Strands готов")
            print(r.json())
            ready = True
            break

    except requests.RequestException:
        pass

    if i % 10 == 0:
        print(f"Ждём модель... {i * 2} сек.")

    time.sleep(2)


if not ready:

    log_file.flush()

    print(
        "\n===== ПОСЛЕДНИЕ СТРОКИ ЛОГА =====\n"
    )

    with open(
        "strands_server.log",
        "r"
    ) as f:
        lines = f.readlines()

    print("".join(lines[-80:]))

Ждём модель... 0 сек.
Ждём модель... 20 сек.
✅ Strands готов
{'status': 'ok', 'model': 'strands-decider-2B-hobson-v21', 'checkpoint': 'StrandsAgents/strands-decider-2B-hobson-v21', 'base_model': 'Qwen/Qwen3.5-2B-Base', 'num_slots': 24, 'max_length': 4096, 'temperature': 0.935220152138837, 'device': 'cuda', 'prefix_cache': True}


In [ ]:
!nvidia-smi

Sat Oct 10 09:26:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P0             26W /   70W |    3819MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
HIRING_INSTRUCTION = """
Вы судья коллегии «Нанимающиеся на работу».

Оцените player_1 и player_2 по всему управленческому поединку.

Главный вопрос:
к кому из этих двух руководителей вы бы пошли работать?

Учитывайте:
- надёжность;
- уважение к людям;
- управленческую твёрдость;
- заботу о команде;
- способность принимать решения;
- способность сохранять устойчивость организации.

Выберите одного игрока.
"""


NEGOTIATION_INSTRUCTION = """
Вы судья коллегии «Отправляющие на переговоры».

Оцените player_1 и player_2 по всему управленческому поединку.

Главный вопрос:
кого из этих двух игроков вы бы отправили вместо себя
на сложные переговоры?

Учитывайте:
- движение к цели;
- способность удерживать управление;
- понимание интересов другой стороны;
- работу с позицией другой стороны;
- способность находить решения;
- сохранение рабочих отношений;
- удержание собственной цели.

Выберите одного игрока.
"""


OWNERSHIP_INSTRUCTION = """
Вы судья коллегии «Доверяющие собственность».

Оцените player_1 и player_2 по всему управленческому поединку.

Главный вопрос:
кому из этих двух игроков вы бы доверили свои деньги,
компанию или другой значимый ресурс?

Учитывайте:
- качество решений;
- управленческую компетентность;
- экономические последствия;
- ответственность;
- отношение к ресурсам;
- риски и последствия действий.

Выберите одного игрока.
"""

In [ ]:
payload = {
    "state": decision_state,

    "questions": {
        "hiring": {
            "type": "choice",
            "instructions": HIRING_INSTRUCTION,
            "criteria": {
                "player_1": "Игрок player_1",
                "player_2": "Игрок player_2"
            }
        },

        "negotiation": {
            "type": "choice",
            "instructions": NEGOTIATION_INSTRUCTION,
            "criteria": {
                "player_1": "Игрок player_1",
                "player_2": "Игрок player_2"
            }
        },

        "ownership": {
            "type": "choice",
            "instructions": OWNERSHIP_INSTRUCTION,
            "criteria": {
                "player_1": "Игрок player_1",
                "player_2": "Игрок player_2"
            }
        }
    }
}

In [ ]:
import requests
import json

response = requests.post(
    "http://127.0.0.1:8000/v1/systemone",
    json=payload,
    timeout=180
)

print("HTTP:", response.status_code)

result = response.json()

print(
    json.dumps(
        result,
        ensure_ascii=False,
        indent=2
    )
)

HTTP: 200
{
  "model": "strands-decider-2B-hobson-v21",
  "answers": {
    "hiring": {
      "type": "choice",
      "choice": "player_1",
      "probabilities": {
        "player_1": 0.6107,
        "player_2": 0.3893
      },
      "confidence": 0.2215
    },
    "negotiation": {
      "type": "choice",
      "choice": "player_1",
      "probabilities": {
        "player_1": 0.5558,
        "player_2": 0.4442
      },
      "confidence": 0.1115
    },
    "ownership": {
      "type": "choice",
      "choice": "player_1",
      "probabilities": {
        "player_1": 0.5509,
        "player_2": 0.4491
      },
      "confidence": 0.1018
    }
  },
  "usage": {
    "input_tokens": 1628,
    "output_tokens": 3
  },
  "latency_ms": 4219.58
}


In [ ]:
for name, answer in result["answers"].items():

    print("\n====================")
    print(name.upper())
    print("====================")

    print(
        json.dumps(
            answer,
            ensure_ascii=False,
            indent=2
        )
    )


HIRING
{
  "type": "choice",
  "choice": "player_1",
  "probabilities": {
    "player_1": 0.6107,
    "player_2": 0.3893
  },
  "confidence": 0.2215
}

NEGOTIATION
{
  "type": "choice",
  "choice": "player_1",
  "probabilities": {
    "player_1": 0.5558,
    "player_2": 0.4442
  },
  "confidence": 0.1115
}

OWNERSHIP
{
  "type": "choice",
  "choice": "player_1",
  "probabilities": {
    "player_1": 0.5509,
    "player_2": 0.4491
  },
  "confidence": 0.1018
}
